In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]
columns = ["visit_id","patient_name","city","department","consultation_fee"]

df = spark.createDataFrame(data, columns)
display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


🔷 2. Write Data as Parquet

In [0]:
df.write \
.mode("overwrite") \
.parquet("/tmp/patient_parquet_parquets")

🔷3. Read Parquet Data

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet_parquets")
display(parquet_df)


visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


🔷4. Schema Inspection

In [0]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)



🔷5. Column Projection (Read Specific Columns)


In [0]:
spark.read.parquet("/tmp/patient_parquet_parquets") \
.select("patient_name","city") \
.show()

+------------+---------+
|patient_name|     city|
+------------+---------+
| Arjun Reddy|Hyderabad|
|  Priya Nair|Bangalore|
|Rahul Sharma|   Mumbai|
|Vikram Singh|  Chennai|
|Sneha Kapoor|    Delhi|
+------------+---------+



🔷6. Filtering Data

In [0]:
spark.read.parquet("/tmp/patient_parquet_parquets") \
.filter("consultation_fee > 3000") \
.show()


+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
+--------+------------+---------+----------+----------------+



🔷7. Partitioned Parquet Write

In [0]:
df.write \
.mode("overwrite") \
.partitionBy("city") \
.parquet("/tmp/patient_parquet_partitioned")

🔷8. Read Partitioned Data



In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned").show()

+--------+------------+-----------+----------------+---------+
|visit_id|patient_name| department|consultation_fee|     city|
+--------+------------+-----------+----------------+---------+
|     103|Rahul Sharma|Dermatology|            1500|   Mumbai|
|     102|Sneha Kapoor|Orthopedics|            3000|    Delhi|
|     101| Arjun Reddy| Cardiology|            5000|Hyderabad|
|     105|Vikram Singh|  Neurology|            7000|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|Bangalore|
+--------+------------+-----------+----------------+---------+



🔷9. Partition Pruning (Performance
Concept)

👉Only that partition is read.

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned") \
.filter("city = 'Hyderabad'") \
.show()


+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+





🔷10. Append Mode

In [0]:
new_data = [
(106,"Ananya Das","Kolkata","Orthopedics",3000)
]
new_df = spark.createDataFrame(new_data, columns)
new_df.write \
.mode("append") \
.parquet("/tmp/patient_parquet_parquets")


🔷11. Overwrite Mode

In [0]:
%sql
CREATE TABLE hive_metastore.default.patient_parquet_table
USING PARQUET LOCATION '/tmp/patient_parquet_parquets';

In [0]:
df.write \
.mode("overwrite") \
.parquet("/tmp/patient_parquet_parquets")

🔷12. Create SQL Table on Parquet

In [0]:
%sql
-- 🔷13. Query Parquet Table

SELECT * FROM patient_parquet_table;

-- 🔷14. Convert Parquet → Delta (Important)

-- CONVERT TO DELTA parquet.`/tmp/patient_parquet`;

-- 🔷15. Compare Parquet vs Delta

-- Try UPDATE (will fail on raw parquet table)

UPDATE patient_parquet_table
SET consultation_fee = 6000
WHERE visit_id = 101;

num_affected_rows
1


In [0]:
# 👉This will fail or not behave as expected

# After conversion to Delta

%sql UPDATE delta.'/tmp/patient_parquet_parquets'
SET consultation_fee = 6000
WHERE visit_id = 101;

In [0]:

# 👉Works perfectly

# 🔷16. Real Use Case Pattern

# Raw Data → Parquet (Landing Zone)
# Clean Data → Delta (Processing Zone)
# Analytics → Delta Tables

# Exercises

# 1. Write DataFrame to Parquet
# 2. Read and display
# 3. Filter high-value records
# 4. Write partitioned Parquet
# 5. Read only one partition
# 6. Append new data
# 7. Create SQL table on Parquet
# 8. Convert to Delta
# 9. Perform update after conversion
# 10. Explain difference between Parquet and Delta